# Un lazo de control al que se le pueden mover las perillas

Un Arduino corre un lazo de control de 500 Hz sobre un motor con un sensor
magnético de ángulo. Este notebook fija las ganancias, corre un experimento y se
trae de vuelta la serie temporal.

Cada celda arranca con `sync_board()`, que recompila el sketch si se lo editó,
lo vuelve a grabar si cambió el binario, y reabre el enlace, lo que resetea la
placa. Así cada celda arranca desde los valores por omisión del sketch, y se
pueden correr en cualquier orden sin preguntarse qué dejó la celda de arriba.

**El motor se va a mover.** Revisar el banco antes de correr cualquier cosa de
las que siguen.

El controlador propiamente dicho está en `../ControlDemo/ControlDemo.ino`. Ése es
el archivo que hay que editar para cambiar la *ley* de control; todo lo de acá
sólo cambia sus parámetros.

---

**Antes de la primera corrida.** Hace falta tener el [Arduino CLI](https://arduino.github.io/arduino-cli/)
en el PATH y el core de AVR instalado:

```
arduino-cli core install arduino:avr
pip install -r ../python/requirements.txt
```

Windows, macOS y Linux funcionan por igual, y la placa se encuentra sola: no
debería hacer falta nombrar ningún puerto. Si el Arduino CLI se instaló con esta
terminal ya abierta, cerrarla y volver a abrirla: el PATH se lee una sola vez al
arrancar, y en Windows ésa es la razón habitual de que `sync_board()` no lo
encuentre.

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt

from bench import *          # sync_board, y las unidades de este equipo

plt.rcParams['figure.figsize'] = (9, 3.5)
plt.rcParams['axes.grid'] = True

## 1. ¿Funciona el hardware?

Conviene correr esto antes que nada, y de nuevo después de tocar el cableado.
Verifica por separado cada parte del equipo —la temporización del lazo, el imán,
el bus I2C, la medición de corriente y por último el motor—, de modo que una
falla apunte a una sola cosa y no a "el experimento no anduvo".

De paso **calibra el cero de la corriente**: con el puente abierto no circula
nada, así que lo que marque el sensor ahí es su offset, y se lo resta. Un cero
corrido se integra en toda medición posterior, y no hay manera de conocerlo salvo
midiéndolo. `dev.zero_current()` lo repite cuando haga falta.

Lo que esa calibración **no** puede darte es la *escala*. El cero se mide porque
hay una condición conocida —el puente abierto—; para la ganancia haría falta una
corriente conocida, y no la hay. Los miliamperes que informa la placa salen de
`SENSE_MV_PER_A` en el sketch, que es un número declarado y no medido. Si importan
como magnitud y no sólo como forma de onda, hay que verificarlo una vez con un
tester en serie con el motor.

Al final acciona el motor un instante para cada lado. Las dos polaridades son
verificaciones distintas porque fallan distinto: que el motor gire dice que el
puente y su alimentación están; que gire *al revés* con el comando invertido es lo
único que distingue `IN1` e `IN2` bien conectados de los dos cambiados, y ese error
no se nota hasta que un lazo cerrado se escapa en vez de establecerse.

Pasar `motor=False` para saltear las dos.

In [ ]:
dev = sync_board()
dev.bringup()

## 2. El sensor

`y_uw` es el ángulo del eje, desenrollado: sigue contando a través de la vuelta
de 4096 cuentas en lugar de saltar de vuelta a cero, así que un eje que gira sin
parar da una recta que sube sin parar. Girar el imán con la mano mientras esto
corre.

`y_uwf` es la misma señal pasada por un filtro de dos polos. `dev.smooth('y', tau)`
fija su constante de tiempo; `tau = 0` lo apaga, que es lo que viene por omisión:
50 ms de retardo es mucha fase para regalar en un lazo de 500 Hz, y siempre se
puede filtrar de este lado después.

In [ ]:
dev = sync_board()
dev.zero()
dev.smooth('y', 0.05)           # 50 ms

df = dev.capture(3.0)

plt.plot(df['t'], df['y_uw'],  lw=0.8, label='y_uw   crudo')
plt.plot(df['t'], df['y_uwf'],          label='y_uwf  filtrado')
plt.xlabel('t [s]'); plt.ylabel('ángulo [grados]'); plt.legend()
plt.title('girar el imán')
plt.show()

print(f'se movió {df["y_uw"].max() - df["y_uw"].min():.1f} grados, '
      f'ruido {df["y_uw"].diff().std():.3f} grados entre muestras')

## 3. El sensor miente, y se puede corregir

El AS5600 no mide el ángulo que uno cree. Un imán apenas descentrado o inclinado
--la hoja de datos pide un cuarto de milímetro-- corre la lectura en una cantidad
que **depende del ángulo** y que se repite vuelta tras vuelta. No es ruido: es una
firma. Y al lazo se le presenta como una ondulación de velocidad que ninguna
ganancia arregla, porque no está en la planta, está en la regla con la que se la
mide.

Medido en este banco, con el imán a la distancia correcta: unos diez grados de
error, casi todo en el segundo armónico de la vuelta.

`calibracion.ipynb` lo mide y arma una tabla de corrección; acá sólo se la carga.
La tabla vive en un archivo de esta computadora y no en la placa, que arranca
siempre sin calibrar: una calibración es una propiedad de *este* banco --este
imán, en este eje-- y no del programa, y una tabla vieja aplicándose en silencio
es peor que ninguna.

`cal` la prende y la apaga en caliente, que es lo que permite ver cuánto sirve en
lugar de suponerlo.

In [ ]:
import os
import calib

dev = sync_board()

RUTA_CAL = 'calibracion.json'

if os.path.exists(RUTA_CAL):
    cal = calib.asegurar(dev, RUTA_CAL)
    picos = ', '.join(f'k={k}: {A:.0f} cuentas'
                      for k, (A, _) in sorted(cal.armonicos.items()))
    print(f'calibracion de {cal.creada}, banco "{cal.banco}"')
    print(f'  armonicos medidos: {picos}')
    print(f'  la tabla corrige hasta {max(abs(v) for v in cal.lut)/8:.0f} cuentas '
          f'= {max(abs(v) for v in cal.lut)/8*calib.GRADOS_POR_CUENTA:.1f} grados')
else:
    cal = None
    print(f'no hay {RUTA_CAL}: correr calibracion.ipynb para medirlo en este banco.')
    print('Lo que sigue igual muestra el error; lo que no va a poder es corregirlo.')

In [ ]:
dev.sfilt = 3                   # el filtro rapido del AS5600; ver calibracion.ipynb

curvas = {}
for etiqueta, valor in (('cal apagada', 0),) + ((('cal prendida', 1),) if cal else ()):
    df = calib.regimen(dev, uff=255, duracion=8.0, cal=valor)
    t, c = calib.serie(df, fuente='y_uw')
    ang, r = calib.residuo(t, c)
    a, prom, _ = calib.promediar(ang, r)
    curvas[etiqueta] = (a, prom * calib.GRADOS_POR_CUENTA)

dev.cal = 0
dev.rest()

# Cada vuelta cae en fases distintas del muestreo, asi que promediar cientos de
# vueltas por caja de angulo reconstruye la forma aunque haya pocas muestras por
# vuelta. Lo que si necesita muchas muestras por vuelta es el AJUSTE de armonicos,
# y por eso la medicion de verdad se hace soltando el motor y no a fondo.
for etiqueta, (a, prom) in curvas.items():
    plt.plot(a, prom, lw=2, label=etiqueta)
    print(f'  {etiqueta:<14} {prom.max() - prom.min():5.2f} grados pico a pico')

plt.axhline(0, color='k', lw=0.8, ls='--')
plt.xlim(0, 4096)
plt.xlabel('angulo dentro de la vuelta [cuentas]')
plt.ylabel('error de angulo [grados]')
plt.title('el mismo error en cada vuelta, y lo que queda al corregirlo')
plt.legend()
plt.show()

## 4. Lazo abierto: ¿qué hace la planta?

`mode = MODE_OPEN` desconecta el controlador y pone `uff` directamente sobre el
motor. Aplicarle un escalón y mirar la respuesta. Esa respuesta —cuán rápido
acelera, cuánta corriente consume— es contra lo que hay que diseñar un
controlador, así que ésta es la primera medición que hay que tomar.

`dev.step()` mantiene `pre` segundos, cambia el parámetro, y después mantiene
`post` segundos más. La placa informa el tick exacto en el que cayó el cambio,
así que `t = 0` es el escalón mismo con precisión de una muestra; la fluctuación
de temporización de este lado nunca entra en los datos.

El comando va de -255 a 255: el signo es el sentido de giro, y el sketch lo aplica
apagando el puente por `ENA` antes de mover `IN1`/`IN2`, así que el cambio de
sentido nunca pasa por un estado conduciendo. `ENA` se modula a 1 kHz, que es lo que
este banco tolera: el L298 cae unos 2 V contra una alimentación de 5 V, y a 20 kHz
lo que se pierde en cada conmutación se lleva un tiempo de encendido que ya venía
escaso -- el motor no llega a arrancar. `dev.pwm(hz)` la cambia, entre 122 Hz y
31,4 kHz, y con un puente MOSFET lo correcto sería subirla.

La velocidad se calcula derivando `y_uwf`, la salida filtrada, y no `y_uw`: derivar
amplifica el ruido, y una cuenta de ruido entre muestras consecutivas son
88 grados por segundo. `dev.smooth('y', tau)` fija el filtro, que vive en la placa
y por lo tanto también es lo que ve el lazo cuando se cierra sobre la posición. La segunda captura es el pedido más
duro que admite el puente —de +200 a -200 de un período al otro—, y sirve para ver
lo que ese cuidado *no* arregla: la corriente que ya circula por el motor no se
puede cortar, y sale por los diodos contra la fuente.

In [ ]:
dev = sync_board()
dev.mode = MODE_OPEN
dev.smooth('y', 0.01)           # 10 ms: la velocidad sale de derivar y_uwf
dev.zero_current()              # el cero del sensor, con el puente abierto

df = dev.step('uff', 200, pre=0.3, post=0.7, back=0)
dev.rest()

por_s = 1 / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'][1:], np.diff(df['y_uwf']) * por_s); a.set_ylabel('velocidad [vueltas/s]')
b.plot(df['t'], df['u']);                         b.set_ylabel('u [pwm]')
c.plot(df['t'], df['i']);                         c.set_ylabel('i [mA]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón en lazo abierto: u = 0 -> 200')
plt.show()

Y ahora el mismo escalón, pero de un sentido al otro. La velocidad tiene que
cruzar el cero y salir del otro lado; el pico de corriente en `t = 0` es la
inductancia del motor descargándose contra la fuente.

In [ ]:
dev = sync_board()
dev.mode = MODE_OPEN
dev.smooth('y', 0.01)
dev.zero_current()
dev.uff  = 200                  # arranca girando para un lado

inv = dev.step('uff', -200, pre=0.3, post=0.7, back=0)
dev.rest()

por_s = 1 / (inv.attrs['dt_us'] * 1e-6) / 360

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(inv['t'][1:], np.diff(inv['y_uwf']) * por_s); a.set_ylabel('velocidad [vueltas/s]')
b.plot(inv['t'], inv['i']);                         b.set_ylabel('i [mA]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('inversión de sentido: u = +200 -> -200')
plt.show()

print(f'pico de corriente {inv["i"].abs().max():.0f} mA, '
      f'contra {inv[inv["t"] < 0]["i"].abs().max():.0f} mA girando parejo')

## 5. Cerrar el lazo sobre la posición

`mode = MODE_PID` pone en marcha el controlador, y `target = POSITION` hace que
trabaje sobre el ángulo. El error es `ref - y_uw`, los dos en cuentas;
`dev.deg()` permite escribir una referencia en grados.

`dev.gains(kp, ki, kd)` toma las ganancias en tiempo continuo —`ki` por segundo,
`kd` en segundos— y las convierte. Los `dev.kp`, `dev.ki` y `dev.kd` propios de
la placa son por *muestra*, que es por lo que su aritmética realmente multiplica.

Con el puente bidireccional el lazo puede corregir para los dos lados, así que
una referencia negativa es tan válida como una positiva y el eje vuelve solo del
lado del que se pase. Contra un puente de un solo cuadrante —`dev.bidir = 0`— el
comando se recorta en cero: el lazo empuja pero no frena, y hay que devolver el
eje a mano.

Este banco tiene una **zona muerta** enorme: medido en lazo abierto, por debajo de
`u ≈ 60` el motor no se mueve, porque el L298 se come unos 2 V de los 5 de
alimentación. Eso decide el ajuste entero.

Por abajo: cualquier `kp` que no llegue a 60 de comando deja el eje parado antes de
la referencia, y ahí se queda. Por arriba: el motor sin carga da 72 vueltas/s a
fondo, así que apenas el comando sale de la zona muerta el eje se va rapidísimo y
`kp` alto oscila. La ventana útil es angosta, y `kp = 0.1` con un `kd` chico está
adentro.

`ki` acá es una trampa, y vale la pena entender por qué: mientras el eje está
parado dentro de la zona muerta el error no baja, así que el integrador se carga
sin límite; cuando por fin rompe la zona muerta el motor arranca a fondo y el
integrador tarda en descargarse. Medido: cualquier `ki` distinto de cero manda el
eje varias vueltas de largo. Es el caso de libro en que la acción integral y una no
linealidad del actuador no se llevan bien.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.smooth('y', 0.005)          # el angulo que realimenta
dev.smooth('e', 0.003)          # y el error que ve el termino derivativo
dev.gains(kp=0.1, ki=0.0, kd=0.002)

dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

df = dev.step('ref', dev.deg(90), pre=0.2, post=0.8, back=0)
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_deg(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['y_uwf'], label='y_uwf')
a.set_ylabel('ángulo [grados]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón en lazo cerrado a 90 grados')
plt.show()

final = df['y_uwf'].iloc[-1]
print(f'terminó en {final:.1f} grados, {90 - final:+.1f} grados de error '
      f'de régimen permanente')

## 6. Barrer una ganancia

La razón de manejar todo esto desde un notebook: cambiar un número, volver a
medir, superponer. Cada pasada es un viaje de ida y vuelta a la placa.

Observar qué compra y qué cuesta subir kp. Con la zona muerta de este banco el
barrido muestra las dos paredes de la ventana útil: los valores chicos ni siquiera
llegan a mover el eje —el comando se queda por debajo de 60— y se paran a medio
camino, y los grandes lo pasan de largo y oscilan.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.smooth('y', 0.005)
dev.smooth('e', 0.003)
dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

corridas = {}
for kp in (0.02, 0.04, 0.06, 0.1):
    dev.gains(kp=kp, kd=0.002)
    corridas[kp] = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)

dev.rest()

for kp, d in corridas.items():
    plt.plot(d['t'], d['y_uwf'], label=f'kp = {kp}')
plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uwf [grados]'); plt.legend()
plt.title('barrido de kp')
plt.show()

for kp, d in corridas.items():
    despues = d[d['t'] > 0]['y_uwf']
    print(f'kp={kp:<7} pico {despues.max():6.1f} grados, '
          f'final {despues.iloc[-1]:6.1f} grados')

## 7. Seguir una rampa

`mode = MODE_RAMP` le suma `refrate` a `ref` en cada período de control, así que
la referencia barre a velocidad constante y el lazo tiene que *seguirla* en lugar
de establecerse.

Un controlador proporcional no puede seguir una rampa sin quedarse atrás: el error
es lo que genera el comando, así que un comando constante necesita un error
constante. En el libro, agregar `ki` es lo que cierra esa brecha; en este banco no,
por lo que dice la sección 5. Así que acá el atraso se mide y se deja, que es la
mitad honesta de la lección.

Las velocidades son 3 y 6 vueltas/s y no 1 y 2: por debajo de unas 2,5 vueltas/s el
comando cae en la zona muerta y el eje avanza a los tirones en lugar de girar.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.smooth('y', 0.005)
dev.smooth('e', 0.003)
dev.gains(kp=0.1, kd=0.002)

dev.zero()
dev.ref     = 0
dev.refrate = dev.rev_per_s(3.0)
dev.mode    = MODE_RAMP

df = dev.step('refrate', dev.rev_per_s(6.0), pre=2.0, post=2.0)
dev.rest()

por_s = 1 / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'][1:], np.diff(dev.as_deg(df['ref'])) * por_s, 'k--', lw=0.8, label='ref')
a.plot(df['t'][1:], np.diff(df['y_uwf']) * por_s, label='y_uwf')
a.set_ylabel('velocidad [vueltas/s]'); a.legend()
b.plot(df['t'], dev.as_deg(df['e'])); b.set_ylabel('e [grados]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('rampa: 3 vueltas/s, después 6')
plt.show()

## 8. P, PI, PD, PID: qué agrega cada término

El escalón de la sección 5 no sirve para separar los términos, porque en este banco
el eje termina parado adentro de la zona muerta y ahí *cualquier* controlador se
queda quieto. La rampa sí sirve: el eje gira sin parar, así que el comando de
régimen es el que hace falta para sostener esa velocidad y no cero, y se lo puede
elegir bien afuera de la zona muerta. A 25 vueltas/s hacen falta unas 120 cuentas
de comando, contra las 60 donde el motor recién arranca.

La tabla del final separa **sesgo** de **dispersión**, y ésa es la mitad de la
lección. El sesgo es cuánto se queda atrás el lazo; la dispersión, cuánto tiembla
alrededor de eso. Son fallas distintas y se arreglan con términos distintos, y
mirar un solo número las confunde: un lazo que oscila fuerte *alrededor* de la
referencia tiene sesgo chico y pasaría por bueno.

Medido en este banco, con las cuatro leyes sobre la misma rampa:

- **P** necesita error para dar comando, así que en teoría se queda atrás una
  cantidad fija, `u_régimen / kp`. Pero acá ni siquiera llega a eso: sin carga el
  lazo oscila con cientos de grados de dispersión, y el sesgo que muestra está
  mezclado con esa oscilación. No está siguiendo la rampa, está rebotando alrededor.
- **PI** tampoco: el integrador aporta comando pero cuesta fase, y sobre un lazo
  que ya venía al límite empeora las dos cosas. Es el que más dispersión hace.
- **PD** es el primero que *sigue*: la dispersión cae a un par de grados, la rampa
  sale lisa. Y recién ahí el atraso aparece por lo que es —unos 300 grados de
  sesgo—, porque la acción derivativa no aporta nada en régimen permanente: en
  régimen el error no cambia. El sesgo de PD es el atraso de P, ahora visible en
  lugar de escondido adentro de una oscilación.
- **PID** agrega el integrador sobre un lazo que ya es estable, y ahí sí hace lo
  que promete: el sesgo cae a menos de un grado sin perder la lisura.

O sea: **kd compra estabilidad, ki compra exactitud, y en esta planta hay que
comprar en ese orden.** Poner ki antes que kd —el paso PI— es lo peor de las dos.

Acá se apoya en que el lazo viene a 500 Hz por omisión (`tickdiv = 10`). A
1 kHz, con seis canales, la fila de telemetría se come el margen y se pierde
alrededor del 20 % de los períodos. Eso es *dropout* aleatorio, y comparar leyes
de control encima de él sería comparar en parte el ruido de muestreo. A 500 Hz no
se pierde ni uno, así que las cuatro corridas se comparan entre sí y nada más.

La planta es ruidosa y sin carga: las corridas no salen idénticas. Lo que se
repite es el orden, no los números.

In [ ]:
dev = sync_board()
dev.zero_current()              # el panel de `i` no dice nada sin esto

LEYES = {'P':   dict(kp=0.02),
         'PI':  dict(kp=0.02, ki=0.02),
         'PD':  dict(kp=0.02,       kd=0.005),
         'PID': dict(kp=0.02, ki=0.2, kd=0.005)}

corridas = {}
for nombre, ganancias in LEYES.items():
    dev.target = POSITION
    dev.smooth('y', 0.01)
    dev.smooth('e', 0.05)
    dev.gains(**ganancias)
    dev.zero()
    dev.ref     = 0
    dev.refrate = 0
    dev.mode    = MODE_RAMP
    corridas[nombre] = dev.step('refrate', dev.rev_per_s(25.0),
                                pre=0.3, post=5, back=0)

dev.rest()

fig, (a, b, c, d, e, f) = plt.subplots(6, 1, sharex=True, figsize=(9, 13))
for nombre, data in corridas.items():
    a.plot(data['t'], dev.as_deg(data['e']), lw=0.9, label=nombre)
    b.plot(data['t'], data['u'],             lw=0.9, label=nombre)
    c.plot(data['t'], data['y_uw'],             lw=0.9, label=nombre)
    c.plot(data['t'], data['y_uwf'],             lw=0.9, label=nombre)
    # d should plot the derivative of y_uwf and y_uw:
    d.plot(data['t'][1:], np.diff(data['y_uwf']) / np.diff(data['t']), lw=0.9, label=f'{nombre} y_uwf')
    d.plot(data['t'][1:], np.diff(data['y_uw']) / np.diff(data['t']), lw=0.9, label=f'{nombre} y_uw')
    e.plot(data['t'], data['ref'],           lw=0.9, label=nombre)
    f.plot(data['t'], data['i'],            lw=0.9, label=nombre)
a.axhline(0, color='k', lw=0.8)
a.set_ylabel('error [grados]'); a.legend(ncol=4, fontsize=8)
d.legend(ncol=4, fontsize=7)
b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
c.set_ylabel('y [grados]'); c.set_xlabel('t [s]')
d.set_ylabel('dy/dt [grados/s]'); d.set_xlabel('t [s]')
e.set_ylabel('ref [grados]'); e.set_xlabel('t [s]')
f.set_ylabel('i [mA]'); f.set_xlabel('t [s]')
for ax in (a, b, c, d, e, f):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('la misma rampa a 25 vueltas/s, cuatro leyes de control')
plt.show()

# El sesgo y la oscilacion son dos cosas distintas y hay que mirarlas por separado:
# la media dice cuanto se queda atras el lazo, la dispersion alrededor de esa media
# dice cuanto tiembla. Un lazo que oscila fuerte alrededor de la referencia tiene
# sesgo chico y dispersion grande, y un solo numero lo haria pasar por bueno.
print(f'{"":4}  {"sesgo":>10}  {"dispersion":>12}  {"pico":>9}    (regimen: t > 2 s)')
for nombre, data in corridas.items():
    regimen = dev.as_deg(data[data['t'] > 2.0]['e'])
    print(f'{nombre:4}  {regimen.mean():+9.1f} deg  {regimen.std():11.1f} deg  '
          f'{dev.as_deg(data["e"]).abs().max():8.0f} deg')

## 9. El mismo controlador, otra señal

`target = CURRENT` cambia la realimentación: el mismo PID ahora trabaja sobre
`ref - i`, la corriente que circula por el motor. Nada más cambia —en el sketch
es una rama en `target_error()`—, así que éste es el mismo controlador contra una
planta mucho más rápida, y necesita otras ganancias exactamente por eso.

`dev.zero_current()` primero: el lazo cierra sobre `ref - i`, así que un cero
corrido en el sensor es un error de régimen permanente que el integrador va a
perseguir sin poder alcanzar. `dev.ma()` escribe la referencia en miliamperes. La corriente es ruidosa, así que
`alpha_i` filtra la medición y `alpha_e` filtra lo que ve el término derivativo;
`dev.smooth()` fija cualquiera de los dos por constante de tiempo.

> **Esta sección necesita una medición de corriente que valga.** El ADC lee A0
> contra la referencia interna de 1,1 V y no contra los 5 V —son 4,5 veces más
> resolución sobre la misma señal, y sin eso la corriente de este motor no llega a
> un escalón de cuantización—. Aun así la señal es chica: conviene mirar qué
> informa `bringup()` en `cero de i` y `calibracion de i` antes de creerle a las
> ganancias que salgan de acá. Y si los miliamperes no cierran con un tester,
> el número a corregir es `SENSE_MV_PER_A`.

In [ ]:
dev = sync_board()

dev.zero_current()
dev.target = CURRENT
dev.smooth('i', 0.005)
dev.smooth('e', 0.010)
dev.gains(kp=0.05, ki=2.0)

dev.ref  = dev.ma(150)
dev.mode = MODE_PID

df = dev.step('ref', dev.ma(300), pre=0.5, post=0.5, back=dev.ma(150))
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_ma(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['i'], label='i')
a.set_ylabel('corriente [mA]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('escalón de corriente, 150 -> 300 mA')
plt.show()

## 10. Qué hace la frecuencia de muestreo

`tickdiv` divide el muestreador de 5 kHz hasta la frecuencia del lazo:
`tickdiv = 10` da los 500 Hz por omisión, `tickdiv = 5` da 1 kHz y `tickdiv = 50`
da 100 Hz. Las ganancias del sketch son
*por muestra*, así que los mismos tres números significan algo distinto a cada
frecuencia, y eso es justamente el punto. Un lazo ajustado a 1 kHz y después
corrido a 100 Hz es un lazo con la décima parte de la acción integral y diez
veces la ganancia derivativa.

`dev.gains()` vuelve a convertir desde tiempo continuo, así que llamarlo de nuevo
después de cambiar `tickdiv` deja el lazo donde estaba. Comentarlo para ver qué
pasa si uno se olvida.

In [ ]:
dev = sync_board()

dev.target = POSITION

for tickdiv in (5, 25, 50):
    dev.tickdiv = tickdiv
    dev.smooth('y', 0.005)            # reconvertidos para el nuevo período
    dev.smooth('e', 0.003)
    dev.gains(kp=0.1, kd=0.002)
    dev.zero()
    dev.ref  = 0
    dev.mode = MODE_PID

    d = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)
    plt.plot(d['t'], d['y_uwf'], label=f'{1e6 / d.attrs["dt_us"]:.0f} Hz')

dev.rest()

plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uwf [grados]'); plt.legend()
plt.title('las mismas ganancias a tres frecuencias de lazo')
plt.show()

## 11. Para terminar

Dejar el motor en reposo y liberar el puerto, o el próximo `sync_board()` lo va a
encontrar ocupado.

In [ ]:
dev = sync_board()
dev.rest()
dev.close()
print('cerrado')